<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb pandas pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [17]:
import pandas as pd

from chromadb import PersistentClient
from pprint import pprint
from pydantic import BaseModel

## Load the dataset

In [9]:
class Book(BaseModel):
  title: str
  author: str
  description: str

In [12]:
PATH_TO_DATASET = "/content/goodreads_dataset.csv"

dataframe = pd.read_csv(PATH_TO_DATASET)
dataframe = dataframe[ dataframe['description'].notna() ] # филтрираме редовете, така че да останат само тези за които имаме дефиниран descriptions

In [13]:
books = [ Book(**record) for record in dataframe.to_dict(orient = "records")]
# title = record["title"], language = recor["language"], ...

In [15]:
pprint(books)

[Book(title="Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", author='J.K. Rowling', description="Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft and Wizardry.After a lifetime of bottling up his magical powers, Harry finally feels like a normal kid. But even within the Wizarding communit

In [16]:
max_description_length = max(len(b.description) for b in books)
print(f"Max description length: {max_description_length}")

Max description length: 2783


## Create ChromaDB collection

In [18]:
PATH_TO_CHROMA = "/content"

chroma_client = PersistentClient(path = PATH_TO_CHROMA)

In [19]:
chroma_collection = chroma_client.get_or_create_collection(name = "goodreads-experiments")

## Load data

In [20]:
ids = [ f"{i+1}" for i in range(len(books))]
metadatas = [{"title": b.title, "author": b.author} for b in books]
documents = [b.description for b in books]

chroma_collection.add(ids=ids, metadatas = metadatas, documents=documents)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 92.0MiB/s]


In [26]:
get_result = chroma_collection.get(include=["metadatas","documents", "embeddings"])

In [29]:
pd.DataFrame({
        "id": get_result["ids"],
        "description": get_result["documents"],
        "metadata": get_result["metadatas"],
        "embeddings": get_result["embeddings"].tolist()
})

,id,description,metadata,embeddings
0,1,Harry Potter's life is miserable. His parents ...,"{'author': 'J.K. Rowling', 'title': 'Harry Pot...","[-0.025321748107671738, 0.0678311288356781, 0...."
1,2,Librarian's note: An alternate cover edition c...,"{'author': 'Markus Zusak', 'title': 'The Book ...","[-0.06795573979616165, 0.02127467840909958, -0..."
2,3,"Could you survive on your own in the wild, wit...","{'title': 'The Hunger Games (The Hunger Games,...","[0.06891398876905441, 0.031229998916387558, 0...."
3,4,Harry Potter is midway through his training as...,{'title': 'Harry Potter and the Goblet of Fire...,"[0.026427675038576126, 0.0033859112299978733, ..."
4,5,Offred is a Handmaid in the Republic of Gilead...,"{'author': 'Margaret Atwood', 'title': 'The Ha...","[-0.023029834032058716, -0.08016648888587952, ..."
...,...,...,...,...
82,83,"In 1920s London, Virginia Woolf is fighting ag...","{'author': 'Michael Cunningham', 'title': 'The...","[-0.010132662020623684, -0.0797799676656723, 0..."
83,84,Pulitzer Prize Winner (1998)In American Pastor...,{'title': 'American Pastoral (The American Tri...,"[0.019527675583958626, 0.004889007192105055, -..."
84,85,"In nineteenth-century China, in a remote Hunan...","{'title': 'Snow Flower and the Secret Fan', 'a...","[-0.10635953396558762, 0.029351145029067993, 0..."
85,86,"Days before his release from prison, Shadow's ...","{'title': 'American Gods (American Gods, #1)',...","[-0.027844298630952835, 0.0008544852025806904,..."


## Query data

In [40]:
from rich.console import Console
from rich.table import Table

def print_query_result(result):
    console = Console()

    queries_count = len(result["ids"])

    for i in range(queries_count):
        table = Table(show_lines = True)
        table.add_column("#")
        table.add_column("Title")
        table.add_column("Author")
        table.add_column("Description")

        results_count = len(result["ids"][i]) # дължината на този масив ни дава отговор колко са резултатите на този масив
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["metadatas"][i][j]["title"], result["metadatas"][i][j]["author"], result["documents"][i][j])

        console.print(table)



In [47]:
query_result = chroma_collection.query(
    query_texts = [
        "Novel about war seen through ordinary people",
        "Find books with magical realism or surreal elements"
    ],
    n_results = 7, # дължината на това което ше трябва да печатаме
    include = ["metadatas", "documents", "distances"]
) # сортирани са по смисъл, а не по id

In [48]:
print_query_result(query_result)

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #  ┃ Title                                       ┃ Author          ┃ Description                                ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 75 │ Regeneration (Regeneration, #1)             │ Pat Barker      │ Regeneration, one in Pat Barker's series   │
│    │                                             │                 │ of novels confronting the psychological    │
│    │                                             │                 │ effects of World War I, focuses on         │
│    │                                             │                 │ treatment methods during the war and the   │
│    │                                             │                 │ story of a decorated English officer sent  │
│    │                                             │                 │ to a military hospital after publicly      │
│    │                                             │                 │ declaring he will no longer fight. Yet the │
│    │                                             │                 │ novel is much more. Written in sparse      │
│    │                                             │                 │ prose that is shockingly clear—the         │
│    │                                             │                 │ descriptions of electrRegeneration, one in │
│    │                                             │                 │ Pat Barker's series of novels confronting  │
│    │                                             │                 │ the psychological effects of World War I,  │
│    │                                             │                 │ focuses on treatment methods during the    │
│    │                                             │                 │ war and the story of a decorated English   │
│    │                                             │                 │ officer sent to a military hospital after  │
│    │                                             │                 │ publicly declaring he will no longer       │
│    │                                             │                 │ fight. Yet the novel is much more. Written │
│    │                                             │                 │ in sparse prose that is shockingly         │
│    │                                             │                 │ clear—the descriptions of electronic       │
│    │                                             │                 │ treatments are particularly harrowing—it   │
│    │                                             │                 │ combines real-life characters and events   │
│    │                                             │                 │ with fictional ones in a work that         │
│    │                                             │                 │ examines the insanity of war like no       │
│    │                                             │                 │ other. Barker also weaves in issues of     │
│    │                                             │                 │ class and politics in this compactly       │
│    │                                             │                 │ powerful book. Other books in the series   │
│    │                                             │                 │ include The Eye in the Door and the Booker │
│    │                                             │                 │ Award winner The Ghost Road.               │
├────┼─────────────────────────────────────────────┼─────────────────┼────────────────────────────────────────────┤
│ 67 │ The Complete Maus                           │ Art Spiegelman  │ Combined for the first time here are Maus  │
│    │                                             │                 │ I: A Survivor's Tale and Maus II - the     │
│    │                                             │    

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #  ┃ Title                                     ┃ Author             ┃ Description                               ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 22 │ Jonathan Strange & Mr Norrell             │ Susanna Clarke     │ Sophisticated, witty, and ingeniously     │
│    │                                           │                    │ convincing, Susanna Clarke's magisterial  │
│    │                                           │                    │ novel weaves magic into a flawlessly      │
│    │                                           │                    │ detailed vision of historical England.    │
│    │                                           │                    │ She has created a world so thoroughly     │
│    │                                           │                    │ enchanting that eight hundred pages leave │
│    │                                           │                    │ readers longing for more.English          │
│    │                                           │                    │ magicians were once the wonder of the     │
│    │                                           │                    │ known world, with fairy servants at their │
│    │                                           │                    │ beck and call; they could cSophisticated, │
│    │                                           │                    │ witty, and ingeniously convincing,        │
│    │                                           │                    │ Susanna Clarke's magisterial novel weaves │
│    │                                           │                    │ magic into a flawlessly detailed vision   │
│    │                                           │                    │ of historical England. She has created a  │
│    │                                           │                    │ world so thoroughly enchanting that eight │
│    │                                           │                    │ hundred pages leave readers longing for   │
│    │                                           │                    │ more.English magicians were once the      │
│    │                                           │                    │ wonder of the known world, with fairy     │
│    │                                           │                    │ servants at their beck and call; they     │
│    │                                           │                    │ could command winds, mountains, and       │
│    │                                           │                    │ woods. But by the early 1800s they have   │
│    │                                           │                    │ long since lost the ability to perform    │
│    │                                           │                    │ magic. They can only write long, dull     │
│    │                                           │                    │ papers about it, while fairy servants are │
│    │                                           │                    │ nothing but a fading memory.But at        │
│    │                                           │                    │ Hurtfew Abbey in Yorkshire, the rich,     │
│    │                                           │                    │ reclusive Mr Norrell has assembled a      │
│    │                                           │                    │ wonderful library of lost and forgotten   │
│    │                                           │                    │ books from England's magical past and     │
│    │                                           │                    │ regained some of the powers of England's  │
│    │                                           │                    │ magicians. He goes to London and raises a │
│    │                                           │      